# NB-02 — Clean Evaluation and Iso-WER Calibration

The ladder is trained. This notebook turns it into something RQ1 can actually be tested with.

It does three jobs:

1. **Clean WER/CER for every checkpoint** on one fixed, untouched test set — the covariate the
   ANCOVA regresses out.
2. **The monotonicity gate.** WER must fall as data rises. If it does not, the data pipeline is
   broken and no amount of attacking will fix it.
3. **Iso-WER matched pairs.** The strongest form of the RQ1 argument: find a *high-data* model
   checkpoint whose clean WER equals a *low-data* model's. Same accuracy, different data volume.
   If their eps50 still differs, that difference cannot be explained by accuracy.

> **Why this matters more than it looks.** Without iso-WER matching, "the 1 h model was easier to
> attack" is answerable with "of course, it is a worse model." The matched pair removes that
> answer entirely. It is the paragraph that converts a reviewer.

---

### Kaggle settings
| Setting | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** (FLEURS cross-corpus arm) |
| Persistence | Files only |
| **Inputs** | `bn-asr-splits` + `bengaliai-speech` + all four `ckpt-xlsr-{1,5,25,100}h` |
| Output | `clean-wer-results` |

Roughly 30–40 min. Cheap compared to everything around it.

## 1 · Boilerplate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys, json, time, re, glob, random, unicodedata, hashlib
import numpy as np, pandas as pd, torch
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
os.makedirs(f"{TEMP}/scratch", exist_ok=True)
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

N_TEST   = 500      # fixed untouched test subset
N_ATTACK = 100      # the Task B attack set; read from the frozen target file
EVAL_BS  = 8
MAX_DUR  = 10.0
RUNGS    = [1, 5, 25, 100]
ISO_TOL  = 0.05     # |WER difference| accepted as an iso-WER match

PROV = {"notebook": "NB-02b-clean-eval-n100", "seed": SEED, "n_test": N_TEST,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
import numpy as _np
NP_BEFORE = _np.__version__
!pip install -q transformers==4.44.2 accelerate==1.13.0 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -2
import subprocess
NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
assert NP_BEFORE == NP_DISK, f"numpy moved {NP_BEFORE} -> {NP_DISK}"
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, jiwer, librosa
print(f"transformers {transformers.__version__} | numpy {NP_BEFORE}")
PROV["transformers"] = transformers.__version__

## 2 · Find every checkpoint: four finals plus the big-rung milestones

In [ ]:
def list_inputs():
    out = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))

INPUTS = list_inputs()

def find_input(pattern, what, required=True):
    hits = [p for p in INPUTS if re.search(pattern, os.path.basename(p), re.I)]
    if not hits:
        if not required: return None
        print("Visible under /kaggle/input:"); [print("   ", p) for p in INPUTS]
        raise AssertionError(f"Could not find {what} (regex {pattern})")
    return max(hits, key=lambda p: (p.count("/"), len(p)))

# ---------------------------------------------------------------------------
# One row per evaluable model. The milestones are the whole point of this
# notebook: they are the only way to get a HIGH-data model down to a LOW-data
# model's accuracy without crippling it artificially.
# ---------------------------------------------------------------------------
models = []
for r in RUNGS:
    root = find_input(rf"ckpt-xlsr-{r}h$", f"ckpt-xlsr-{r}h", required=False)
    if root is None:
        print(f"  rung {r}h NOT ATTACHED -- skipping"); continue
    fin = root if os.path.exists(f"{root}/config.json") else f"{root}/final"
    assert os.path.exists(f"{fin}/config.json"), f"no config.json under {fin}"
    models.append(dict(rung=r, kind="final", step=None, path=fin))
    for m in sorted(glob.glob(f"{root}/milestones/step-*")):
        if os.path.exists(f"{m}/config.json"):
            models.append(dict(rung=r, kind="milestone",
                               step=int(m.rsplit("-", 1)[1]), path=m))

print(f"\n{len(models)} evaluable checkpoints:")
for m in models:
    print(f"   rung {m['rung']:>3}h  {m['kind']:<9} "
          f"{'step '+str(m['step']) if m['step'] else '':<10} {m['path']}")
assert len(models) >= 4, "Attach all four ckpt-xlsr-*h datasets."

## 3 · The evaluation sets

In [ ]:
SPLITS = find_input(r"splits", "bn-asr-splits")
ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

def fix_paths(df):
    """NB-00 baked in ITS session's mount point; re-resolve if it moved."""
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0): return df
    stem = os.path.basename(os.path.dirname(p0)); ext = os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}", f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, f"audio dir '{stem}' not found -- attach bengaliai-speech"
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df

# TEST, not dev: dev drove early stopping, so it is no longer untouched.
test = pd.read_csv(f"{SPLITS}/test.csv")
test = test[test.duration <= MAX_DUR].sample(N_TEST, random_state=SEED).reset_index(drop=True)
test = fix_paths(test)

# ---------------------------------------------------------------------------
# THE 100-UTTERANCE ATTACK SET.
#
# Read the ids from the ladder's OWN frozen target file so the clean WER is
# measured on exactly the utterances Task B attacked, in manifest order
# (original 25 first). Never re-sample here: a fresh sample would silently be a
# different set and the iso-WER match would not correspond to the eps50 runs.
# ---------------------------------------------------------------------------
TGT = next((os.path.join(p, "task_b_targets_n100.csv") for p in INPUTS
            if os.path.exists(os.path.join(p, "task_b_targets_n100.csv"))), None)
assert TGT, ("task_b_targets_n100.csv not found. Attach ONE of the Task B result "
             "datasets, e.g. zoayriaabedin/eps50-rung100-milestone-3500-t5-n100.")
TGT_SHA = hashlib.sha256(open(TGT, "rb").read()).hexdigest()
print(f"targets  : {TGT}")
print(f"targets sha256: {TGT_SHA}")
assert TGT_SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "This is NOT the target file the six Task B runs used, so a clean WER measured "
    "here would not correspond to the attacked set.")

tg = pd.read_csv(TGT)
assert len(tg) == 100 and tg.utt_id.nunique() == 100, f"target file has {len(tg)} rows"

_full = pd.read_csv(f"{SPLITS}/test.csv")
attack = tg[["utt_id"]].merge(_full, on="utt_id", how="left")
assert attack["path"].notna().all(), (
    "some of the 100 attack utterances are not in test.csv -- wrong splits version")
assert (attack.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), (
    "transcript mismatch between the frozen target file and test.csv")
attack = fix_paths(attack)

N25 = 25   # the original RQ1 spine is the FIRST 25 rows of the manifest.
           # Row order in the eps50 RESULT csvs is duration-sorted, not manifest
           # order -- that distinction already cost one wrong analysis. Here we
           # are reading the TARGET file, which is in manifest order.

print(f"test set   : {len(test)} utts | {test.duration.sum()/60:.1f} min")
print(f"attack set : {len(attack)} utts | mean {attack.duration.mean():.1f}s "
      f"| first {N25} = the published RQ1 spine")
assert os.path.exists(test["path"].iloc[0]), "test audio unresolved"
assert os.path.exists(attack["path"].iloc[0]), "attack audio unresolved"


In [ ]:
# FLEURS-bn: an INDEPENDENT corpus with different speakers and recording
# conditions. This is the mitigation promised in NB-00 for the fact that
# Bengali.AI ships no speaker IDs, so our splits are recording-disjoint rather
# than speaker-disjoint. If the ladder ordering holds here too, speaker leakage
# cannot be what produced it.
FLEURS = None
try:
    from datasets import load_dataset
    ds = load_dataset("google/fleurs", "bn_in", split="test",
                      streaming=True, trust_remote_code=True)
    rows = []
    for s in ds:
        a, sr0 = s["audio"]["array"], s["audio"]["sampling_rate"]
        d = len(a) / sr0
        if 1.0 < d <= MAX_DUR:
            w = (librosa.resample(np.asarray(a, np.float32), orig_sr=sr0, target_sr=16000)
                 if sr0 != 16000 else np.asarray(a, np.float32))
            rows.append({"wav": w, "text": norm_bn(s.get("transcription") or
                                                   s.get("raw_transcription"))})
        if len(rows) >= 200: break
    FLEURS = rows
    print(f"FLEURS-bn: {len(FLEURS)} utterances loaded")
except Exception as e:
    print(f"FLEURS unavailable ({type(e).__name__}) -- cross-corpus arm skipped.")
    print("Not fatal, but the speaker-leakage mitigation is then unsupported.")

## 4 · Evaluation

In [ ]:
def load_wavs(paths):
    out = []
    for p in paths:
        try:
            a, _ = librosa.load(p, sr=16000, mono=True, duration=MAX_DUR)
            a = np.asarray(a, np.float32)
            pk = np.abs(a).max()
            out.append(a / pk if pk > 1.0 else a)
        except Exception:
            out.append(np.zeros(16000, np.float32))
    return out


@torch.no_grad()
def transcribe(model, processor, wavs, bs=EVAL_BS):
    """Length-sorted batched greedy CTC decode. Order is restored at the end."""
    order = np.argsort([len(w) for w in wavs])
    hyps = [None] * len(wavs)
    do_norm = bool(getattr(processor.feature_extractor, "do_normalize", True))
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]
        chunk = [wavs[j] for j in idx]
        T = max(len(w) for w in chunk)
        x = torch.zeros(len(chunk), T, device=DEVICE)
        m = torch.zeros(len(chunk), T, device=DEVICE)
        for k, w in enumerate(chunk):
            x[k, :len(w)] = torch.from_numpy(w).to(DEVICE); m[k, :len(w)] = 1
        if do_norm:
            n = m.sum(-1, keepdim=True).clamp(min=1)
            mu = (x*m).sum(-1, keepdim=True)/n
            var = (((x-mu)*m)**2).sum(-1, keepdim=True)/n
            x = ((x-mu)/torch.sqrt(var+1e-7))*m
        ids = torch.argmax(model(input_values=x, attention_mask=m.long()).logits, -1)
        for k, j in enumerate(idx):
            hyps[j] = norm_bn(processor.batch_decode(ids[k:k+1])[0])
    return hyps


def score(refs, hyps):
    pairs = [(norm_bn(r), h) for r, h in zip(refs, hyps) if str(r).strip()]
    if not pairs: return 1.0, 1.0
    R = [p[0] for p in pairs]; H = [p[1] for p in pairs]
    return float(jiwer.wer(R, H)), float(jiwer.cer(R, H))


print("caching audio once, reused across all checkpoints...")
t0 = time.time()
TEST_WAVS   = load_wavs(test["path"].tolist())
ATTACK_WAVS = load_wavs(attack["path"].tolist())   # 100 clips
print(f"  {len(TEST_WAVS)+len(ATTACK_WAVS)} clips in {time.time()-t0:.0f}s")

In [ ]:
rows = []
for i, m in enumerate(models):
    t0 = time.time()
    proc = Wav2Vec2Processor.from_pretrained(m["path"])
    mdl  = Wav2Vec2ForCTC.from_pretrained(m["path"]).float().to(DEVICE).eval()
    mdl.config.apply_spec_augment = False

    w_test, c_test = score(test["text"], transcribe(mdl, proc, TEST_WAVS))
    # ONE transcription pass over the 100 clips; score the full set and the
    # original-25 subset from it, so the n=25 replication check is free.
    hyp_atk = transcribe(mdl, proc, ATTACK_WAVS)
    w_atk,  c_atk  = score(attack["text"], hyp_atk)
    w_a25,  c_a25  = score(attack["text"].iloc[:N25].tolist(), hyp_atk[:N25])

    w_fl = c_fl = None
    if FLEURS:
        w_fl, c_fl = score([r["text"] for r in FLEURS],
                           transcribe(mdl, proc, [r["wav"] for r in FLEURS]))

    rows.append(dict(rung=m["rung"], kind=m["kind"], step=m["step"], path=m["path"],
                     wer_test=w_test, cer_test=c_test,
                     wer_attack=w_atk, cer_attack=c_atk,
                     wer_attack25=w_a25, cer_attack25=c_a25,
                     wer_fleurs=w_fl, cer_fleurs=c_fl))
    lbl = f"rung {m['rung']}h {m['kind']}" + (f" step-{m['step']}" if m["step"] else "")
    print(f"[{i+1}/{len(models)}] {lbl:<28} test WER {w_test:.4f} | attack100 {w_atk:.4f} | attack25 {w_a25:.4f}"
          + (f" | FLEURS {w_fl:.4f}" if w_fl is not None else "")
          + f"   ({time.time()-t0:.0f}s)")
    del mdl; torch.cuda.empty_cache()

res = pd.DataFrame(rows)

## 5 · The monotonicity gate

**WER must fall as training data rises.** If it does not, something is wrong with the data
pipeline, and no amount of attacking will produce a meaningful result. Stop here if this fails.

In [ ]:
fin = res[res.kind == "final"].sort_values("rung").reset_index(drop=True)
print(fin[["rung", "wer_test", "cer_test", "wer_attack", "wer_fleurs"]].to_string(index=False))

mono = bool((fin.wer_test.diff().dropna() < 0).all())
print(f"\nmonotonic on the test set: {mono}")
if not mono:
    print("\n*** GATE FAILED. Do not proceed to NB-05. ***")
    print("    Check: rung nesting in NB-00, per-rung dev WER in NB-01, and whether")
    print("    a rung was cut off by the step cap before it could separate.")
else:
    print(f"spread: {fin.wer_test.iloc[0]:.4f} (1h) -> {fin.wer_test.iloc[-1]:.4f} "
          f"({fin.rung.iloc[-1]}h) = {fin.wer_test.iloc[0]/fin.wer_test.iloc[-1]:.2f}x")

if FLEURS:
    mono_fl = bool((fin.wer_fleurs.diff().dropna() < 0).all())
    print(f"monotonic on FLEURS (cross-corpus): {mono_fl}")
    print("  -> if this also holds, speaker leakage in Bengali.AI cannot explain the ladder,")
    print("     which is the mitigation NB-00 promised for the missing speaker IDs.")
    PROV["monotonic_fleurs"] = mono_fl

PROV["monotonic_test"] = mono

## 6 · Iso-WER matching

For each low-data rung, find the **high-data checkpoint with the same clean WER**. Those pairs
share accuracy but differ in training data, so any eps50 gap between them is attributable to
data volume alone — not to one model simply being worse.

In [ ]:
# ---------------------------------------------------------------------------
# MATCH ON THE ATTACK SET, NOT THE TEST SET.
#
# eps50 is measured on the 100 attack utterances, so the accuracy that must be
# held fixed is each model's WER on THOSE clips -- not its WER on the
# 500-utterance test set. The two disagree enough to change the answer:
# matching on test WER pairs 1h with a 25h milestone (25x data), while matching
# on attack WER pairs it with a 100h milestone (100x data) at a SMALLER gap.
# Same rigour, four times the contrast.
#
# MATCH_ON stays configurable so the thesis can report both and show the
# conclusion does not depend on the choice.
# ---------------------------------------------------------------------------
MATCH_ON = "wer_attack"        # now the 100-utterance set. "wer_test" = secondary

def build_pairs(key):
    out = []
    for _, lo in fin.iterrows():
        cand = res[res.rung > lo.rung].copy()
        if cand.empty: continue
        cand["gap"] = (cand[key] - lo[key]).abs()
        best = cand.nsmallest(1, "gap").iloc[0]
        out.append(dict(
            match_on=key,
            low_rung=int(lo.rung), low_wer=round(float(lo[key]), 4), low_path=lo.path,
            high_rung=int(best.rung), high_kind=best.kind,
            high_step=(int(best.step) if pd.notna(best.step) else None),
            high_wer=round(float(best[key]), 4), high_path=best.path,
            wer_gap=round(float(best.gap), 4),
            usable=bool(best.gap <= ISO_TOL),
            data_ratio=round(float(best.rung / lo.rung), 1)))
    return pd.DataFrame(out)

iso      = build_pairs(MATCH_ON)
iso_alt  = build_pairs("wer_test" if MATCH_ON == "wer_attack" else "wer_attack")

cols = ["low_rung","low_wer","high_rung","high_kind","high_step","high_wer",
        "wer_gap","usable","data_ratio"]
print(f"=== PRIMARY: matched on {MATCH_ON} ===")
print(iso[cols].to_string(index=False))
print(f"\n=== SECONDARY (robustness check): matched on {iso_alt.match_on.iloc[0]} ===")
print(iso_alt[cols].to_string(index=False))

good = iso[iso.usable]
print(f"\n{len(good)}/{len(iso)} primary pairs matched within {ISO_TOL} WER")
if len(good):
    print("\nThese are the comparisons that make RQ1 airtight. NB-05 must measure")
    print("eps50 on BOTH members of each pair -- including the MILESTONE checkpoints,")
    print("not just the four finals:")
    for _, p in good.iterrows():
        tag = f"{p.high_rung}h {p.high_kind}" + (f" step-{p.high_step}" if p.high_step else "")
        print(f"   {p.low_rung}h final (WER {p.low_wer}) vs {tag} (WER {p.high_wer})"
              f"   -- {p.data_ratio:.0f}x more data at the same accuracy")
    print("\nNB-05 run list implied by these pairs:")
    need = sorted({(int(p.low_rung), "final") for _, p in good.iterrows()} |
                  {(int(p.high_rung),
                    "final" if p.high_kind == "final" else f"milestone-{int(p.high_step)}")
                   for _, p in good.iterrows()})
    for r, ck in need:
        print(f"   RUNG={r:<4} CHECKPOINT={ck}")
else:
    print(f"\nNo pair within {ISO_TOL}. Widen ISO_TOL and carry the residual as a covariate.")


## 7 · Save

In [ ]:
res.to_csv(f"{WORK}/clean_wer_all_checkpoints_n100.csv", index=False)
fin.to_csv(f"{WORK}/ladder_final_wer.csv", index=False)
if len(iso):
    iso.to_csv(f"{WORK}/iso_wer_pairs.csv", index=False)
    iso_alt.to_csv(f"{WORK}/iso_wer_pairs_secondary.csv", index=False)

PROV.update(
    n_checkpoints=len(res),
    ladder={int(r.rung): round(float(r.wer_test), 4) for _, r in fin.iterrows()},
    ladder_cer={int(r.rung): round(float(r.cer_test), 4) for _, r in fin.iterrows()},
    attack_set_wer={int(r.rung): round(float(r.wer_attack), 4) for _, r in fin.iterrows()},
    attack_set_wer25={int(r.rung): round(float(r.wer_attack25), 4) for _, r in fin.iterrows()},
    targets_sha256=TGT_SHA, n_attack=int(len(attack)),
    n_iso_pairs=int(iso.usable.sum()) if len(iso) else 0,
    iso_tol=ISO_TOL, match_on=MATCH_ON,
)
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
json.dump({"title": "clean-wer-results-n100", "id": "YOURUSERNAME/clean-wer-results-n100",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print(json.dumps(PROV, indent=2))
PUB = {1: 0.6471, 5: 0.4755, 25: 0.3235, 100: 0.2500}
print("\n=== REPLICATION CHECK: attack-set WER on the original 25 ===")
print(f"{'rung':>6s}{'published':>11s}{'measured 25':>13s}{'shift':>8s}{'n=100':>9s}")
for _, r in fin.iterrows():
    p = PUB.get(int(r.rung))
    if p is None: continue
    print(f"{int(r.rung):>5d}h{p:>11.4f}{r.wer_attack25:>13.4f}"
          f"{r.wer_attack25-p:>+8.4f}{r.wer_attack:>9.4f}")
print("  A shift beyond about 0.02 means the checkpoint or the audio is not the")
print("  one the published numbers came from -- stop and find out why.")
print("\nPublish as dataset 'clean-wer-results-n100', then use it in NB-09.")

---

## Fill this into RUNLOG.md

| Rung | test WER | test CER | attack-set WER | FLEURS WER |
|---|---|---|---|---|
| 1 h | | | | |
| 5 h | | | | |
| 25 h | | | | |
| 100 h | | | | |

### Gates
- [ ] **Monotonic on the test set** — if this fails, stop; do not run NB-05
- [ ] **Monotonic on FLEURS** — supports the speaker-leakage mitigation
- [ ] **At least one usable iso-WER pair** — needed for the strong form of RQ1

### Then
**NB-05**, once per `(RUNG, EDIT_TYPE)`. Start with `RUNG=1, T5` and `RUNG=100, T5` — the two
extremes. If eps50 does not separate there, the middle rungs will not rescue it, and you will
have learned that in ~2 hours instead of ~5.